In [13]:
%pwd

'd:\\Medical_chatbot\\research'

In [14]:
import os
os.chdir("../")

In [15]:
%pwd

'd:\\Medical_chatbot'

In [16]:
import imp
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter



C:\Users\manojkrbahuguna.SDD-26\AppData\Local\Temp\ipykernel_20708\4270841984.py:1: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp


In [17]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    document = loader.load()
    return document


In [18]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [20]:
len(extracted_data)

637

In [22]:
from telnetlib import DO
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs:List[Document])-> List[Document]:
    """ List of document containing Source and content only"""
    minimal_docs: List[Document] =[]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content= doc.page_content,
                metadata= {"source": src}
            )
        )

    return minimal_docs



In [23]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [26]:
# Split the documents in smaller chunks

from langchain_community.document_loaders.base_o365 import CHUNK_SIZE


def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap= 20
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [27]:
text_chunks = text_split(minimal_docs)
print(len(text_chunks))

5859


In [28]:
from ast import mod
from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(
        model_name = model_name
    )
    return embeddings

embeddings = download_embedding()

In [3]:
embeddings

HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ', embed_instruction='', show_progress=False)

In [ ]:
vector = embeddings.embed_query("mohit is a good boy")
vector

In [29]:
from dotenv import load_dotenv
import os
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [30]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
pc

In [31]:
from pinecone import ServerlessSpec
from pinecone.deprecation_warnings import create_index
index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name= index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")

    )
index = pc.Index(index_name)

In [32]:
from langchain_core import documents
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name

) 

## Adding more document to same Index

In [33]:
from importlib.metadata import metadata


dswith = Document(
    page_content="Alzheimer's is a progressive neurodegenerative disorder and the most common form of dementia, characterized by memory loss, cognitive decline, and behavioral changes due to brain-damaging protein buildup (amyloid plaques/tau tangles). Early signs include forgetting recent information, misplacing items, and confusion, progressing through mild, moderate, and severe stages. No cure exists, but treatments focus on managing symptoms.",
    metadata={"source":"Wikipedia"} 
)



In [34]:
docsearch.add_documents(documents=[dswith])

['f45938ce-1fc0-4cfc-9bc9-e6306549676e']

In [35]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [36]:
retrieved_docs = retriever.invoke("What is Alzhimer")
retrieved_docs

[Document(id='bf1951ce-f86b-4508-98d0-71273a1fe2be', metadata={'source': 'data\\Medical_book.pdf'}, page_content='(NOAH). 1530 Locust St., #29, Philadelphia, PA 19102-\n4415. (800) 473-2310. <http://www.albinism.org>.\nCarol A. Turkington\nAlbuterol see Bronchodilators\nAlcohol-related \nneurologic disease\nDefinition\nAlcohol, or ethanol, is a poison with direct toxic\neffects on nerve and muscle cells. Depending on which\nnerve and muscle pathways are involved, alcohol can\nhave far-reaching effects on different parts of the brain,\nperipheral nerves, and muscles, with symptoms of mem-'),
 Document(id='97377a9b-1d57-4c45-9776-972290a434e0', metadata={'source': 'data\\Medical_book.pdf'}, page_content='cate the likelihood of ALD because the disease is carried\non the X-chromosome by the female lineage of families.\nTreatment\nTreatment for all forms of ALD consists of treating\nthe symptoms and supporting the patient with physical\ntherapy, psychological counseling, and special educati

In [37]:
from langchain_openai import ChatOpenAI

chatmodel = ChatOpenAI(model="gpt-4o")

In [43]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [44]:
question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [46]:
response = rag_chain.invoke({"input":"what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by abnormal release of a chemical from the pituitary gland, leading to increased growth in bone and soft tissue along with various body disturbances. Gigantism refers to a similar condition occurring in children, resulting from excess growth hormone before the closure of growth plates, leading to unusually large growth. Both conditions are characterized by an excess of growth hormone.
